In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_csv_df
from src.plotting import set_plotting_style

In [ ]:
# Already prepared accent palette applying using function from mine src module (plotting)
set_plotting_style()

In [ ]:
DF_PATH = "../data/raw/raw-data.csv"

# Safely loading dataframe using function from mine src module (data)
df = load_csv_df(
    df_path=DF_PATH,
)

In [ ]:
df.info()
"""Dataframe overview:
-------------------------
- Shape: 195 rows and 10 columns
- Data types: 9 numerical features (float64) and 1 categorial features (str)
- Features: some features name is non-standard, they include spaces, special characters, and numbers
(e.g., % water, % of world population, HDI, Land in km2)
"""

In [ ]:
display(df.head(5))
display(df.tail(5))
"""Dataframe observation:
----------------------------
- 'Country name' acts as a unique row identifier and should be moved to index
- Features like 'Population' and 'Total in km2' contain massive numbers.
While '% water', '% of world population' and 'HDI' are decimal. Scaler
is strictly necessary.
"""

In [ ]:
country_unique_values = df["Country name"].unique()
print(f"Country name unique: {len(country_unique_values)} values")
"""Country name observation:
--------------------------------
- As i expected, 'Country name' is just identifier and
has 195 unique values (195 rows in dataframe summary).
"""

In [ ]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)
display(df.describe().T)

print(f"HDI median: {df['HDI'].median():.3f}")
print(f"% water median: {df['% water'].median():.3f}")
print(f"% water sum: {df['% water'].sum():.2f}%") # % water not in world but in specified country
"""Dataframe observation:
-------------------------
- Already see NaN in count (e.g., 'Fertility rate' missing 4 values, summary row count is 195 but
in feature is 191 instead, all features have at least 1 missing value exclude 'Total in km2').
- 'Total in km2' has correct outliers (Country have huge difference in territory).
The same applies to 'Population' and 'Population density'.
- '% water' highly skewed towards 0 (median is only 1.4%).
- 'HDI' and 'Fertility rate' have reasonable ranges but contain minor missing values.
"""

In [ ]:
df_dup = df.duplicated().sum() # 0 duplicates
print(f"Duplicated rows: {df_dup}")

In [ ]:
df_missing = df.isna().sum()

missing_data = pd.DataFrame({
    "missing_count": df_missing,
    "missing_percent": (df_missing / len(df)) * 100
}).reset_index().rename(columns={'index': 'feature'})

missing_data = missing_data[missing_data['missing_count'] > 0].sort_values(by='missing_percent', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(
    data=missing_data,
    x="missing_percent",
    y="feature"
)
plt.title("Missing Count Distribution")
plt.xlabel("Percentage %")
plt.ylabel("Features")
plt.tight_layout()
plt.grid(True)
plt.show()
"""Missing values observation:
---------------------------------
- Missing values are in 'HDI', 'Fertility rate', 'Land in km2', 'Population',
'Inland water in km2', '% water', 'Population density' and '% of world population'
(8 features summary).
- Missing data is minimal across all dataframe (under 3%).
- The most missing values are in 'HDI' (2.5% of summary df shape).
"""

In [ ]:
skewness = df.select_dtypes("number").skew()
display(skewness.sort_values(ascending=False))
"""Dataframe features skewness observation:
-----------------------------------
- 'Population density' has the largest skew (11.05) and 'Population' have as well (8.32).
- 'Fertility rate' has moderate positive skew (1.07).
- 'HDI' is only one normally distributed feature with slight left skew (-0.41).
"""

In [ ]:
num_cols = df.select_dtypes("number").columns

n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(
        y=df[col],
        ax=axes[i]
    )
    axes[i].set_title(f"Boxplot ({col})")
    axes[i].set_ylabel("Values")

for i in range(len(num_cols), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    center=0
)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()
"""Dataframe heatmap observation:
-------------------------------------
- 'Total in km2' and 'Land in km2' obviously duplicated features (corr = 1.00).
'Population' and '% of world Population' also repeated and only distract model in future.
- 'Island water in km2' and 'Total in km2' have strong synergy (corr = 0.81).
- 'HDI' and 'Fertility rate' have strong invertible synergy (corr = -0.85).
They show different meanings - they higher the level of human development
in a country, the lower the fertility rate.
"""

In [ ]:
neg_val = (df.select_dtypes('number') < 0).sum()
neg_cols = neg_val[neg_val > 0]
print(f"Negative: {len(neg_cols)} columns") # Dataframe doesn't have negative values.

water_over_100 = (df["% water"] > 100).any()
print(f"Water > 100%: {water_over_100}") # Dataframe water percentage doesn't exceed 100.

calculated_density = df["Population"] / df["Land in km2"]
density_diff = (df["Population density"] - calculated_density).abs()
print(f"Max density diff: {density_diff.max():.3f}") # Have some differences (160.730)

In [ ]:
check_df = df.copy()
check_df["calc_density_land"] = check_df["Population"] / check_df["Land in km2"]
check_df["calc_density_total"] = check_df["Population"] / check_df["Total in km2"]

check_df["diff"] = (check_df["Population density"] - check_df["calc_density_land"]).abs()
check_df = check_df.sort_values(by="diff", ascending=False)

display(check_df[
    ["Country name", "Population density", "calc_density_land", "calc_density_total", "diff"]
])
"""Dataframe validation:
-----------------------------
- 'Population density' matches 'calc_density_total' across dataframe.
That means, density was calculated by total land instead of specific country,
explaining maximum difference (160.730).
- No negative values were found across numeric features.
- Water percentage values are logical and within valid boundaries (<= 100%).
"""

# 📌 EDA Summary & ML Preprocessing

---

## 🔍 Key Exploration Data Analysis

### 1. Data Structure
* **Dataframe shape:** 195 rows and 10 columns (9 numerical, 1 categorical).
* **Identifier:** `Country name` contains exact same 195 unique values and acts as a row identifier. It will be moved to index.
* **Scale disparity:** Massive scale differences exist - values from decimals (`% water`, `HDI`) to millions (`Population`, `Total in km2`).

---

### 2. Data Quality & logical validation
* **Duplicates and negatives:** 0 duplicated values found and no negative values across numerical columns.
* **Missing values:** minor missings detected in 8 numerical columns. All under ***2.5%*** (across all dataframe) and
highest in `HDI`.
* **Percentage and formula validation:** `% water` strictly satisfies 100%. `Population density` matches perfectly
with my `calc_density_total`. Density was calculated by total area instead of specified country.

---

### 3. Data Distribution & Relationships
* **Skewness and outliers:** High right-skewness observed in `Population density` ($11.05$) and `Population` ($8.32$) due extreme values.
`HDI` is the only near normal distributed feature ($-0.41$).
* **Multicollinearity:** Perfect coincidence between `Total in km2` and `Land in km2` ($1.00$), as well as
`Population` and `% of world population`.
* **Domain relationships:** Strong negative correlation between `HDI` and `Fertility rate` (&-0.85$),
aligning global demographic trends.

---

## 🛠️ ML Preprocessing

| Step                 | Target feature | EDA observation                     | Action                                  |
|:---------------------| :--- |:------------------------------------|:----------------------------------------|
| **1. Indexing**      | `Country name` | Unique categorical identifier       | `df.set_index('Country name')`          |
| **2. Imputation**    | 8 numerical columns | Minor NaNs ($< 2.5%$)               | Fill missing values with column medians |
| **3. Drop repeats**  | `Total in km2`, `% of world population` | High multicollinearity ($1.00$) | Drop to avoid model distraction & feature overweighting |
| **4. Log Transform** | Skewed features (`Population`, `Density`, etc.) | Extreme right-skewness       | Apply `np.log1p()` transformation          |
| **5. Normalization**      | All numerical features | Unmatched metric units and scales       | Apply `StandardScaler()`     |

> ***EDA*** stage successfully finalized. Ready for pipeline execution.